## data transformation and preparation

**spliting the data (train/test)**

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

data = pd.read_csv('../data/student_mental_health.csv')

data = data.drop(columns=['Student_ID'])

X = data.drop("Mental_Health_Risk", axis=1)
y = data['Mental_Health_Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)




**handle missing values**

In [ ]:
from sklearn.impute import SimpleImputer



num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

nums = X_train[num_cols]

q1 = nums.quantile(0.25)
q3 = nums.quantile(0.75)
iqr = q3 - q1


lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

outliers = ((nums < lower_limit) | (nums > upper_limit)).sum()

print(outliers)

**Outlier Decision:**  
Outliers detected via IQR (e.g., low sleep hours, low attendance) represent realistic extreme student behaviors rather than data entry errors, so we keep them to avoid hiding high-risk students. Their impact on scale will be handled cleanly by `RobustScaler` inside the preprocessing pipeline.

**encode target (y)**

In [ ]:
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
y_train = y_train.map(risk_map)
y_test = y_test.map(risk_map)

**pipeline**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder


num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

num_pipeline = Pipeline([
    ('imputer', num_imputer),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline([
    ('imputer', cat_imputer),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

### Justification of Preprocessing Decisions

1. **Train/Test Split (80/20 + Stratified):**
   - Split first before any transformation to strictly prevent data leakage, using `stratify=y` to keep identical class proportions in train and test sets.

2. **Missing Value Imputation:**
   - **Numerical features (`median`):** Robust against outliers compared to the mean.
   - **Categorical features (`most_frequent`):** Standard strategy for filling missing text categories without inventing new labels.

3. **Outlier Treatment:**
   - Outliers detected via IQR represent realistic student behaviors (e.g., very low sleep or attendance) rather than data errors, so they are kept to preserve high-risk signals.

4. **Categorical & Target Encoding:**
   - **`OneHotEncoder` for features:** Converts categories into binary columns without imposing an artificial numeric order; `handle_unknown='ignore'` prevents crashes on unseen categories.
   - **Manual mapping for target (`Low: 0, Medium: 1, High: 2`):** Preserves the natural severity order of risk.

5. **Scaling (`RobustScaler`):**
   - Scales numerical features using median and IQR instead of mean/std, limiting the distortion caused by outliers.

6. **Class Imbalance (No SMOTE):**
   - From EDA, `Mental_Health_Risk` is balanced (~34% High, 33% Medium, 33% Low), so oversampling/undersampling techniques like SMOTE are unnecessary.

7. **Pipeline (`ColumnTransformer` + `Pipeline`):**
   - All preprocessing steps are bundled and fitted exclusively on `X_train`, then applied to `X_test` to guarantee zero data leakage and easy reuse during deployment.